In [1]:
import sys
import os
sys.path.append("../")
sys.path.append("../../")

In [2]:
from scale_rl.common.wandb_utils import *

In [3]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE

In [4]:
# entity = 'draftrec'
# project_name = 'crossq'

# run_exp_names_to_analysis_exp_names = {
#     "HalfCheetah-v4": "CrossQ",
#     "Hopper-v4": "CrossQ",
#     "Walker2d-v4": "CrossQ",
#     "Ant-v4": "CrossQ",
#     "Humanoid-v4": "CrossQ",
# }
# run_exp_names = list(run_exp_names_to_analysis_exp_names.keys())
# metrics = ['eval/mean_reward']

In [5]:
# def _convert_wandb_df_to_eval_df(runs_df, metrics: List, n_samples=100000):
#     """
#     Collect the history of specified metrics from wandb runs.

#     Parameters:
#     runs_df: pandas DataFrame
#         DataFrame containing the wandb runs.
#     metrics: list
#         List of metric names to collect history for.
#     n_samples: int, optional
#         Number of samples to fetch from the run history. Default is 100000.

#     Returns:
#     eval_df: pandas DataFrame with following columns
#         exp_name / env_name / seed / metric / env_step / value
#     """
#     records = []

#     for idx in tqdm(range(len(runs_df))):
#         run_data = runs_df.iloc[idx]
#         exp_name = run_data["exp_name"]
#         config = run_data["config"]
#         if "env_name" in config:
#             env_name = config["env_name"]
#         elif "env" in config:
#             env_name = config["env"]["env_name"]
#         else:
#             raise ValueError
#         seed = config["seed"]

#         run = run_data["run"]
#         history = run.history(samples=n_samples)
#         steps = history["global_step"]

#         for metric in metrics:
#             if metric in history.columns:
#                 metric_history = history[metric].dropna()
#                 for idx, val in metric_history.items():
#                     env_step = steps[idx]

#                     records.append(
#                         {
#                             "exp_name": exp_name,
#                             "env_name": env_name,
#                             "seed": seed,
#                             "metric": metric,
#                             "env_step": env_step,
#                             "value": val,
#                         }
#                     )

#     return pd.DataFrame(records)

In [6]:
# runs = collect_runs(entity=entity, project_name=project_name)
# for run in tqdm(runs):
#     run.config["group_name"] = 'CrossQ'
#     run.config['seed'] = int(run.config['seed'])
#     run.config["env_name"] = run.config['env']
#     run.config["exp_name"] = run.config['env']
    

In [7]:
# filtered_runs = filter_runs(runs, exp_names = run_exp_names)
# wandb_df = convert_runs_to_dataframe(
#     runs = filtered_runs, 
#     run_exp_name_to_analysis_exp_name=run_exp_names_to_analysis_exp_names
# )
# wandb_df = wandb_df[wandb_df.apply(lambda row: 'finished' in str(row['run']), axis=1)]
# eval_df = _convert_wandb_df_to_eval_df(wandb_df, metrics)
# eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')
# eval_df


In [8]:
# eval_df['metric'] = eval_df['metric'].map({'eval/mean_reward': 'avg_return'})
# eval_df['env_step'] = eval_df['env_step'].astype(int)
# eval_df['env_step'].iloc[0] = 0
# eval_df

In [9]:
# eval_df.to_csv("../results/0106_crossq_mujoco_1m.csv")

In [10]:
eval_df = pd.read_csv('../results/0106_crossq_mujoco_1m.csv', index_col=0)

In [11]:
MUJ_STEPS = 1e6 # 1M

In [12]:
_eval_df = eval_df
overall_df = []
for env_name in MUJOCO_ALL:
    # if env_name == "Humanoid-v4":
    #     continue
    env_df = _eval_df[_eval_df["env_name"] == env_name]
    assert max(env_df["env_step"]) == MUJ_STEPS
    env_df = env_df[env_df["env_step"] == MUJ_STEPS]
    num_seeds = env_df["seed"].nunique()

    grouped_data = env_df.groupby("env_step")["value"]

    env_steps = grouped_data.mean().index.values
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)

    print(f"{env_name} - number of seeds: {num_seeds}")
    print(f"{round(mean)} [{round(mean - 1.96 * std_err)}, {round(mean + 1.96 * std_err)}]")
    
    overall_df.append(env_df)
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

HalfCheetah-v4 - number of seeds: 10
12893 [11771, 14015]
Hopper-v4 - number of seeds: 10
2467 [1855, 3079]
Walker2d-v4 - number of seeds: 10
6257 [5277, 7237]
Ant-v4 - number of seeds: 10
6980 [6834, 7126]
Humanoid-v4 - number of seeds: 10
10480 [10307, 10653]
Overall: 7815.291 ± 1053.729


In [13]:
_eval_df = normalize_score_with_random_and_base_score(eval_df, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE)
overall_df = []
for env_name in MUJOCO_ALL:
    # if env_name == "Humanoid-v4":
    #     continue
    env_df = _eval_df[_eval_df["env_name"] == env_name]
    assert max(env_df["env_step"]) == MUJ_STEPS
    env_df = env_df[env_df["env_step"] == MUJ_STEPS]
    num_seeds = env_df["seed"].nunique()

    grouped_data = env_df.groupby("env_step")["value"]

    env_steps = grouped_data.mean().index.values
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)

    print(f"{env_name} - number of seeds: {num_seeds}")
    print(f"{round(mean, 3)} ± {round(1.96 * std_err, 3)}")
    
    overall_df.append(env_df)
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

HalfCheetah-v4 - number of seeds: 10
1.213 ± 0.103
Hopper-v4 - number of seeds: 10
0.763 ± 0.191
Walker2d-v4 - number of seeds: 10
1.586 ± 0.249
Ant-v4 - number of seeds: 10
1.757 ± 0.036
Humanoid-v4 - number of seeds: 10
2.054 ± 0.034
Overall: 1.475 ± 0.141


In [14]:
from rliable import library as rly
from rliable import metrics as rly_metrics
from rliable import plot_utils as rly_plot_utils

aggregate_func = lambda x: np.array([
  rly_metrics.aggregate_mean(x),
  rly_metrics.aggregate_median(x),
  rly_metrics.aggregate_iqm(x),
  rly_metrics.aggregate_optimality_gap(x/1000)])

In [15]:
_eval_df = normalize_score_with_random_and_base_score(eval_df, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE)
metric_matrix_dict = generate_metric_matrix_dict(
    _eval_df, 
    env_step=MUJ_STEPS, 
    metric_type="avg_return",
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)

In [16]:
_aggregate_scores = aggregate_scores["CrossQ"]
_aggregate_score_cis = aggregate_score_cis["CrossQ"]
print(_aggregate_score_cis[0])
for i, score in enumerate(["Mean", "Median", "IQM"]):
    print(f"{score}: {round(_aggregate_scores[i], 3)} [{round(_aggregate_score_cis[0][i], 3)}, {round(_aggregate_score_cis[1][i], 3)}]")

[1.32983923 1.31716472 1.3943004  0.99839162]
Mean: 1.475 [1.33, 1.608]
Median: 1.489 [1.317, 1.643]
IQM: 1.565 [1.394, 1.71]
